# 04. Tools & Structured Outputs

The model does NOT call your backend directly. Tool calling is a mechanism where the model **proposes** a structured action, and your application **validates, authorizes, and executes** it.

In this lab, we will build an impenetrable execution boundary for a customer support bot that can retrieve orders and issue refunds. We will explicitly test that no malicious payload can bypass business rules or authorization.

## Part 1 & 2 — Raw JSON Schema vs Pydantic
We define our tool arguments strictly using Pydantic. We explicitly forbid extra fields and enforce strict constraints (like positive amounts in cents).

In [1]:
import json
from typing import Optional, Literal, Set, Dict, Any
from pydantic import BaseModel, Field, ConfigDict, ValidationError
from enum import Enum

class RefundReason(str, Enum):
    DAMAGED = 'damaged'
    LOST = 'lost'
    CUSTOMER_REQUEST = 'customer_request'

class GetOrderArgs(BaseModel):
    model_config = ConfigDict(extra='forbid')
    order_id: str

class IssueRefundArgs(BaseModel):
    model_config = ConfigDict(extra='forbid')
    order_id: str = Field(..., description='The unique order identifier.')
    amount_cents: int = Field(..., gt=0, description='Amount to refund in cents (must be positive).')
    reason: RefundReason = Field(..., description='The approved reason for the refund.')
    idempotency_key: str = Field(..., description='Unique key to prevent duplicate refunds.')

print('Generated JSON Schema for IssueRefundArgs:')
print(json.dumps(IssueRefundArgs.model_json_schema(), indent=2))

Generated JSON Schema for IssueRefundArgs:
{
  "$defs": {
    "RefundReason": {
      "enum": [
        "damaged",
        "lost",
        "customer_request"
      ],
      "title": "RefundReason",
      "type": "string"
    }
  },
  "additionalProperties": false,
  "properties": {
    "order_id": {
      "description": "The unique order identifier.",
      "title": "Order Id",
      "type": "string"
    },
    "amount_cents": {
      "description": "Amount to refund in cents (must be positive).",
      "exclusiveMinimum": 0,
      "title": "Amount Cents",
      "type": "integer"
    },
    "reason": {
      "$ref": "#/$defs/RefundReason",
      "description": "The approved reason for the refund."
    },
    "idempotency_key": {
      "description": "Unique key to prevent duplicate refunds.",
      "title": "Idempotency Key",
      "type": "string"
    }
  },
  "required": [
    "order_id",
    "amount_cents",
    "reason",
    "idempotency_key"
  ],
  "title": "IssueRefundArgs",
  "ty

## Part 3 — Typed Result Models
We also type the outputs of our tools, returning predictable models rather than arbitrary dicts.

In [2]:
class OrderResult(BaseModel):
    order_id: str
    customer_id: str
    total_cents: int
    status: str

class RefundResult(BaseModel):
    status: Literal['refund_issued', 'already_processed']
    transaction_id: str
    amount_cents: int
    order_id: str

class ErrorResult(BaseModel):
    error_type: Literal['SCHEMA_ERROR', 'BUSINESS_ERROR', 'AUTH_ERROR', 'UNKNOWN_TOOL']
    message: str


## Part 4 — Execution Context
The untrusted model NEVER provides its own identity or permissions. The application injects the active Execution Context into the dispatcher.

In [3]:
class ExecutionContext(BaseModel):
    user_id: str
    tenant_id: str
    roles: Set[str]

ctx_valid_agent = ExecutionContext(user_id='agent_007', tenant_id='tnt_1', roles={'support_agent'})
ctx_unauthorized = ExecutionContext(user_id='hacker_99', tenant_id='tnt_1', roles={'guest'})


## Part 5 — Robust Tool Registry & Backend Implementations
We register the schema, the underlying python callable, the effect type (read/write), and the required permission.

In [4]:
# Mock Database
DB_ORDERS = {
    'ORD-123': {'customer_id': 'agent_007', 'total_cents': 5000, 'status': 'delivered'},
    'ORD-456': {'customer_id': 'other_person', 'total_cents': 10000, 'status': 'delivered'}
}
DB_PROCESSED_REFUNDS = set()

# Backend Implementations
def get_order_impl(args: GetOrderArgs) -> OrderResult:
    if args.order_id not in DB_ORDERS:
        raise ValueError(f'Order {args.order_id} not found.')
    rec = DB_ORDERS[args.order_id]
    return OrderResult(order_id=args.order_id, customer_id=rec['customer_id'], total_cents=rec['total_cents'], status=rec['status'])

def issue_refund_impl(args: IssueRefundArgs) -> RefundResult:
    # Idempotency check
    if args.idempotency_key in DB_PROCESSED_REFUNDS:
        return RefundResult(status='already_processed', transaction_id='tx_existing', amount_cents=args.amount_cents, order_id=args.order_id)
    
    DB_PROCESSED_REFUNDS.add(args.idempotency_key)
    return RefundResult(status='refund_issued', transaction_id='tx_new_999', amount_cents=args.amount_cents, order_id=args.order_id)

# Tool Registry
TOOL_REGISTRY = {
    'get_order': {'schema': GetOrderArgs, 'func': get_order_impl, 'effect': 'read', 'permission': 'any'},
    'issue_refund': {'schema': IssueRefundArgs, 'func': issue_refund_impl, 'effect': 'write', 'permission': 'support_agent'}
}
print('Tool Registry initialized.')

Tool Registry initialized.


## Part 6 — The Safe Dispatcher
The dispatcher enforces the boundary. It NEVER calls a registered function with unvalidated raw args.

In [5]:
def dispatch_tool(tool_name: str, raw_args: str, ctx: ExecutionContext) -> BaseModel:
    # 1. Tool Existence
    if tool_name not in TOOL_REGISTRY:
        return ErrorResult(error_type='UNKNOWN_TOOL', message=f"Tool '{tool_name}' does not exist.")
    
    entry = TOOL_REGISTRY[tool_name]
    
    # 2. Authorization
    req_perm = entry['permission']
    if req_perm != 'any' and req_perm not in ctx.roles:
        return ErrorResult(error_type='AUTH_ERROR', message='Permission denied. You lack the required role.')
        
    # 3. Schema Validation
    try:
        validated_args = entry['schema'].model_validate_json(raw_args)
    except ValidationError as e:
        # Sanitize error for the model
        err_msg = ', '.join([f"{err['loc'][0]}: {err['msg']}" for err in e.errors()])
        return ErrorResult(error_type='SCHEMA_ERROR', message=err_msg)

    # 4. Business Validation (Domain Specific)
    if tool_name == 'issue_refund':
        if validated_args.order_id not in DB_ORDERS:
            return ErrorResult(error_type='BUSINESS_ERROR', message='Order not found.')
        order = DB_ORDERS[validated_args.order_id]
        if order['customer_id'] != ctx.user_id and 'admin' not in ctx.roles:
            return ErrorResult(error_type='BUSINESS_ERROR', message='Cannot refund another customer\'s order.')
        if validated_args.amount_cents > order['total_cents']:
            return ErrorResult(error_type='BUSINESS_ERROR', message='Refund amount exceeds order total.')
            
    # 5. Execute
    try:
        return entry['func'](validated_args)
    except Exception as e:
        return ErrorResult(error_type='BUSINESS_ERROR', message=str(e))


## Part 7 — Notebook Tests (Validation Suite)
We explicitly test the boundary to ensure the model cannot bypass our rules.

In [6]:
import pandas as pd

tests = [
    {'desc': '1. Success', 'ctx': ctx_valid_agent, 'tool': 'issue_refund', 'args': '{"order_id": "ORD-123", "amount_cents": 1000, "reason": "damaged", "idempotency_key": "key1"}'},
    {'desc': '2. Idempotency (Duplicate)', 'ctx': ctx_valid_agent, 'tool': 'issue_refund', 'args': '{"order_id": "ORD-123", "amount_cents": 1000, "reason": "damaged", "idempotency_key": "key1"}'},
    {'desc': '3. Other-customer order', 'ctx': ctx_valid_agent, 'tool': 'issue_refund', 'args': '{"order_id": "ORD-456", "amount_cents": 1000, "reason": "damaged", "idempotency_key": "key2"}'},
    {'desc': '4. Refund > total', 'ctx': ctx_valid_agent, 'tool': 'issue_refund', 'args': '{"order_id": "ORD-123", "amount_cents": 9000, "reason": "damaged", "idempotency_key": "key3"}'},
    {'desc': '5. Unknown tool', 'ctx': ctx_valid_agent, 'tool': 'drop_table', 'args': '{}'},
    {'desc': '6. Malicious tenant_id injection', 'ctx': ctx_valid_agent, 'tool': 'get_order', 'args': '{"order_id": "ORD-123", "tenant_id": "tnt_admin"}'},
    {'desc': '7. Authorization denied', 'ctx': ctx_unauthorized, 'tool': 'issue_refund', 'args': '{"order_id": "ORD-123", "amount_cents": 1000, "reason": "damaged", "idempotency_key": "key4"}'}
]

results = []
for t in tests:
    res = dispatch_tool(t['tool'], t['args'], t['ctx'])
    results.append({'Test': t['desc'], 'Result Type': type(res).__name__, 'Output': res.model_dump_json()})

display(pd.DataFrame(results))

,Test,Result Type,Output
0,1. Success,RefundResult,"{""status"":""refund_issued"",""transaction_id"":""tx..."
1,2. Idempotency (Duplicate),RefundResult,"{""status"":""already_processed"",""transaction_id""..."
2,3. Other-customer order,ErrorResult,"{""error_type"":""BUSINESS_ERROR"",""message"":""Cann..."
3,4. Refund > total,ErrorResult,"{""error_type"":""BUSINESS_ERROR"",""message"":""Refu..."
4,5. Unknown tool,ErrorResult,"{""error_type"":""UNKNOWN_TOOL"",""message"":""Tool '..."
5,6. Malicious tenant_id injection,ErrorResult,"{""error_type"":""SCHEMA_ERROR"",""message"":""tenant..."
6,7. Authorization denied,ErrorResult,"{""error_type"":""AUTH_ERROR"",""message"":""Permissi..."


## Part 8 — Bounded Correction Loop
We allow the model to correct Schema errors (like omitting required fields), but Authorization errors are explicitly NOT sent back to the model for retry.

In [7]:
def simulate_correction_loop(tool_name: str, raw_args: str, ctx: ExecutionContext):
    max_retries = 2
    for attempt in range(max_retries):
        print(f'Attempt {attempt+1}...')
        res = dispatch_tool(tool_name, raw_args, ctx)
        if isinstance(res, ErrorResult):
            if res.error_type == 'SCHEMA_ERROR':
                print('-> Schema error detected. Model receives error and tries to fix...')
                # Mock the model fixing the args
                raw_args = '{"order_id": "ORD-123", "amount_cents": 1000, "reason": "damaged", "idempotency_key": "key_fix"}'
            elif res.error_type == 'AUTH_ERROR':
                print('-> Auth error detected. Hard halt! Do not send back for retry.')
                break
            else:
                print(f"-> Business error: {res.message}. Stopping.")
                break
        else:
            print('-> Success!', res)
            break

print('--- Recoverable Schema Error ---')
simulate_correction_loop('issue_refund', '{"order_id": "ORD-123"}', ctx_valid_agent)
print('\n--- Unrecoverable Auth Error ---')
simulate_correction_loop('issue_refund', '{"order_id": "ORD-123", "amount_cents": 1000, "reason": "damaged", "idempotency_key": "key_x"}', ctx_unauthorized)

--- Recoverable Schema Error ---
Attempt 1...
-> Schema error detected. Model receives error and tries to fix...
Attempt 2...
-> Success! status='refund_issued' transaction_id='tx_new_999' amount_cents=1000 order_id='ORD-123'

--- Unrecoverable Auth Error ---
Attempt 1...
-> Auth error detected. Hard halt! Do not send back for retry.


## Part 9 — Structured Outputs (Separate from Tool Calling)
Returning a final structured object (e.g. a `SupportDecision`) is semantically different from calling a tool to execute a backend action.

In [8]:
from typing import Literal

class SupportDecision(BaseModel):
    category: Literal['refund', 'replacement', 'escalate']
    summary: str
    requires_human: bool

model_final_output = '{"category": "refund", "summary": "Approved refund for damaged item.", "requires_human": false}'
decision = SupportDecision.model_validate_json(model_final_output)
print('Final Decision Object:', decision)

Final Decision Object: category='refund' summary='Approved refund for damaged item.' requires_human=False


## Parts 10 & 11 — Optional Real OpenAI Integration
Here we test real Tool Calling and Real Structured Outputs via the `openai` SDK, hooking it strictly into our safe `dispatch_tool`.

In [9]:
import os
api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    print('No OPENAI_API_KEY found. Skipping real API call.')
else:
    from openai import OpenAI
    client = OpenAI(api_key=api_key)
    
    # --- 10. Real Read-Only Tool Call ---
    print("\n--- Real OpenAI Tool Calling ---")
    tools = [{
        'type': 'function',
        'function': {
            'name': 'get_order',
            'description': 'Retrieves details about a specific customer order.',
            'parameters': GetOrderArgs.model_json_schema()
        }
    }]
    messages = [{'role': 'user', 'content': 'Can you check the status of ORD-123?'}]
    
    response = client.chat.completions.create(model='gpt-4o-mini', messages=messages, tools=tools)
    msg = response.choices[0].message
    messages.append(msg)
    
    if msg.tool_calls:
        tc = msg.tool_calls[0].function
        print(f"Model proposes tool: {tc.name} | Args: {tc.arguments}")
        
        # DISPATCH THROUGH SECURE BOUNDARY
        res = dispatch_tool(tc.name, tc.arguments, ctx_valid_agent)
        print(f"Dispatcher Result: {res}")
        
        messages.append({
            'role': 'tool',
            'tool_call_id': msg.tool_calls[0].id,
            'name': tc.name,
            'content': res.model_dump_json() # Returns structured Pydantic JSON
        })
        
        final_response = client.chat.completions.create(model='gpt-4o-mini', messages=messages)
        print("\nFinal Answer:", final_response.choices[0].message.content)

    # --- 11. Real Structured Outputs ---
    print("\n--- Real OpenAI Structured Outputs (.parse) ---")
    try:
        response2 = client.beta.chat.completions.parse(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': 'I want a refund for my broken TV!'}],
            response_format=SupportDecision,
        )
        decision_obj = response2.choices[0].message.parsed
        print('Parsed SupportDecision Object:', decision_obj)
    except AttributeError:
        print('Please update the openai SDK to use .parse() for structured outputs.')

No OPENAI_API_KEY found. Skipping real API call.
